# ML Assignment 02

### Dataset: House Price Prediction Dataset

### Total Marks: 100

---

## Exam Instructions:

1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights


## Student Information


In [ ]:
# Fill in your information
name = "Rafiur Rahman Saikat"  # Write your full name here
email = "md.rafiur.rahman.saikkat@gmail.com"  # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

---

## Question 1 (10 Marks)

Load the House Price dataset and display:

- Dataset shape
- First 10 rows
- 5 random samples


In [ ]:
# Question 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File handling / loading dataset
file_path = "house_price_regression_dataset.csv"
if not os.path.exists(file_path):
    try:
        import kagglehub

        downloaded_path = kagglehub.dataset_download("prokshitha/home-value-insights")
        for fname in os.listdir(downloaded_path):
            if fname.endswith(".csv"):
                file_path = os.path.join(downloaded_path, fname)
                break
    except Exception:
        pass

df = pd.read_csv(file_path)

# 1. Display dataset shape
print("Dataset Shape:")
print(df.shape)

# 2. Display first 10 rows
print("\nFirst 10 Rows:")
display(df.head(10))

# 3. Display 5 random samples
print("\n5 Random Samples:")
display(df.sample(5, random_state=42))

---

## Question 2 (10 Marks)

Handle missing values and perform feature engineering:

- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.


In [ ]:
# Question 2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Step 1: Drop columns with more than 50% missing values
missing_threshold = 0.50 * len(df)
df_cleaned = df.dropna(thresh=missing_threshold, axis=1)

# Separate features (X) and target (y)
X = df_cleaned.drop(columns=["House_Price"])
y = df_cleaned["House_Price"]

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# Step 2: Impute missing numerical values using SimpleImputer with mean strategy
if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy="mean")
    X[num_cols] = num_imputer.fit_transform(X[num_cols])

# Step 3: Impute missing categorical values using most frequent strategy
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# Step 4: Perform train-test split with test_size=0.2 and random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Display the shape of final train and test sets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

---

## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):

- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).


In [ ]:
# Question 3
# Extract Square_Footage as feature (X) and House_Price as target (y)
X_train_sf = X_train["Square_Footage"].values
y_train_sf = y_train.values
X_test_sf = X_test["Square_Footage"].values
y_test_sf = y_test.values

# Feature Scaling (Standardization) to enable gradient descent convergence
mean_x = np.mean(X_train_sf)
std_x = np.std(X_train_sf)

X_train_scaled = (X_train_sf - mean_x) / std_x
X_test_scaled = (X_test_sf - mean_x) / std_x

# Initialize slope (m) and intercept (c)
m_scaled = 0.0
c_scaled = 0.0
learning_rate = 0.01
epochs = 1000
n = len(X_train_scaled)

# Batch Gradient Descent
for _ in range(epochs):
    y_pred_iter = m_scaled * X_train_scaled + c_scaled
    dm = (-2 / n) * np.sum(X_train_scaled * (y_train_sf - y_pred_iter))
    dc = (-2 / n) * np.sum(y_train_sf - y_pred_iter)
    m_scaled -= learning_rate * dm
    c_scaled -= learning_rate * dc

# Convert back to original unscaled feature domain: y = m * X + c
m = m_scaled / std_x
c = c_scaled - (m_scaled * mean_x / std_x)

# Predict values for the test set
y_test_pred = m * X_test_sf + c

# Print the learned m and c values
print(f"Learned Slope (m)    : {m:.4f}")
print(f"Learned Intercept (c): {c:.4f}")

print("\nFirst 5 Predicted Values on Test Set:")
print(np.round(y_test_pred[:5], 2))
print("First 5 Actual Values on Test Set:")
print(np.round(y_test_sf[:5], 2))

---

## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:

- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`


In [ ]:
# Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define features
num_features = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
cat_features = ["Neighborhood_Quality"]

# Build ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ],
    remainder="drop",
)

print("ColumnTransformer successfully created:")
print(column_transformer)

---

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:

- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples


In [ ]:
# Question 5
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Build Pipeline with ColumnTransformer and SGDRegressor
pipeline_sgd = Pipeline(
    steps=[
        ("preprocessor", column_transformer),
        ("regressor", SGDRegressor(random_state=42)),
    ]
)

# Train the pipeline
pipeline_sgd.fit(X_train, y_train)

# Evaluate using RMSE and R² score on test set
y_pred_sgd = pipeline_sgd.predict(X_test)
rmse_sgd = np.sqrt(mean_squared_error(y_test, y_pred_sgd))
r2_sgd = r2_score(y_test, y_pred_sgd)

print(f"SGDRegressor RMSE    : {rmse_sgd:.4f}")
print(f"SGDRegressor R² Score: {r2_sgd:.4f}")

# Print predicted vs actual values for the first 10 test samples
comparison_sgd = pd.DataFrame(
    {
        "Actual House Price": y_test.iloc[:10].values,
        "Predicted House Price": y_pred_sgd[:10],
    }
)
display(comparison_sgd)

---

## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:

- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples


In [ ]:
# Question 6
from sklearn.linear_model import LinearRegression

# Build Pipeline with ColumnTransformer and LinearRegression
pipeline_lr = Pipeline(
    steps=[("preprocessor", column_transformer), ("regressor", LinearRegression())]
)

# Train the pipeline
pipeline_lr.fit(X_train, y_train)

# Evaluate using RMSE and R² score on test set
y_pred_lr = pipeline_lr.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"LinearRegression RMSE    : {rmse_lr:.4f}")
print(f"LinearRegression R² Score: {r2_lr:.4f}")

# Print predicted vs actual values for the first 10 test samples
comparison_lr = pd.DataFrame(
    {
        "Actual House Price": y_test.iloc[:10].values,
        "Predicted House Price": y_pred_lr[:10],
    }
)
display(comparison_lr)

---

## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)

### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:

- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)


In [ ]:
# Question 7
# 1. Pick random 100 datas from the dataset
sample_100 = df.sample(n=100, random_state=42)

# Features: Square_Footage, Num_Bedrooms, Num_Bathrooms
# Target  : House_Price
X_sample = sample_100[["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]].values


y_sample = sample_100["House_Price"].values

# 2. Prepare design matrix X by adding bias term (x0 = 1)
X_b = np.c_[np.ones((len(X_sample), 1)), X_sample]

# 3. Compute coefficients using the Normal Equation: θ = (XᵀX)⁻¹ Xᵀy
theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y_sample)

# 4. Print the learned coefficients (θ values)
print("Learned Coefficients (θ values):")
print(f"θ0 (Intercept)      : {theta[0]:.4f}")
print(f"θ1 (Square_Footage) : {theta[1]:.4f}")
print(f"θ2 (Num_Bedrooms)   : {theta[2]:.4f}")
print(f"θ3 (Num_Bathrooms)  : {theta[3]:.4f}")